# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [7]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/AhmedElwhaiby/FlyRank-Inten-Repo"
REPO_DIR = "FlyRank-Inten-Repo"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd, numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Ranking / scoring. The question is "which pages first," not "is this page good or bad"
(classification) or "what kinds of pages exist" (clustering) — it's a priority order for
a reviewer with limited capacity, so the output is a ranked score, not a label.

In [8]:
# If this were classification, we'd expect a ready-made yes/no label to predict.
# There isn't one for "should this page be reviewed" -- confirming this is a
# scoring/ranking problem, not classification.

candidate_label_cols = [c for c in df.columns if "review" in c.lower() or "priority" in c.lower()]
print("Pre-existing review/priority label columns found:", candidate_label_cols)

Pre-existing review/priority label columns found: []


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Target (proxy, not a raw label): a gap score = observed CTR (or engagement) minus the
expected value for that page's position tier and intent. This is OBSERVED, not
rule-defined — ctr and avg_position are measured, not derived from someone's threshold.
The tier-expected baseline is a lookup, not the label itself.

In [9]:
# The target is a gap: observed ctr minus the expected ctr for that page's position tier.
# Show the expected baseline is genuinely computable from OBSERVED columns (not a rule label).
valid = df[df["avg_position"] > 0].copy()
valid["position_tier"] = pd.cut(
    valid["avg_position"], bins=[0, 3, 10, 20, 1000],
    labels=["1-3", "4-10", "11-20", "21+"]
)
expected_ctr_by_tier = valid.groupby("position_tier", observed=True)["ctr"].median()
print("Expected (median) CTR by position tier:")
print(expected_ctr_by_tier)

valid["ctr_gap"] = valid["ctr"] - valid["position_tier"].map(expected_ctr_by_tier)
print("\nSample gap scores:")
print(valid[["avg_position", "position_tier", "ctr", "ctr_gap"]].head())

Expected (median) CTR by position tier:
position_tier
1-3      0.00
4-10     0.16
11-20    0.10
21+      0.00
Name: ctr, dtype: float64

Sample gap scores:
   avg_position position_tier   ctr  ctr_gap
0          10.6         11-20  0.76     0.66
1          20.3           21+  0.05     0.05
2          36.5           21+  0.09     0.09
3           6.2          4-10  0.49     0.33
4          44.0           21+  0.13     0.13


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Precision@K, where K = the reviewer's realistic weekly capacity (e.g. top 20-50). This
matches how the output is actually used — the reviewer only ever looks at the top K, so
accuracy across all pages doesn't reflect the real cost of a wrong call.

In [10]:
# Sketch of precision@K -- how "good" would be measured once we have real flags to check against.
# (Real evaluation needs reviewer-confirmed labels; this just shows the mechanic.)
K = 30
ranked = valid.sort_values("ctr_gap").head(K)  # most negative gap = most under-performing first
print(f"Top {K} pages by CTR gap (candidates a reviewer would see first):")
print(ranked[["avg_position", "position_tier", "ctr", "ctr_gap"]])

# precision@K would be: (# of these K that a reviewer confirms are worth fixing) / K
# left as a placeholder until reviewer-labeled outcomes exist

Top 30 pages by CTR gap (candidates a reviewer would see first):
       avg_position position_tier  ctr  ctr_gap
24029           4.7          4-10  0.0    -0.16
18791           6.4          4-10  0.0    -0.16
18796          10.0          4-10  0.0    -0.16
11826           4.6          4-10  0.0    -0.16
11828           7.4          4-10  0.0    -0.16
11829           7.1          4-10  0.0    -0.16
11837           9.9          4-10  0.0    -0.16
11846           3.9          4-10  0.0    -0.16
6324            6.6          4-10  0.0    -0.16
18789           7.3          4-10  0.0    -0.16
18822           7.0          4-10  0.0    -0.16
2996            7.8          4-10  0.0    -0.16
2998            7.2          4-10  0.0    -0.16
3000            7.4          4-10  0.0    -0.16
3001            7.0          4-10  0.0    -0.16
3002            5.5          4-10  0.0    -0.16
3010            7.9          4-10  0.0    -0.16
3011            7.0          4-10  0.0    -0.16
27129           8.0    

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [11]:
# One row = one visible content page (pseudonymized), with its 90-day CTR/position/engagement metrics.
print("Rows:", len(df))
print("Columns:", len(df.columns))
df.head()

Rows: 30000
Columns: 44


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A flat rule ("CTR < X% is bad") is already wrong on its face — CTR at position 1 and
CTR at position 9 aren't comparable, so a single threshold can't work. The minimum fix
is a lookup table (expected CTR per position tier), not ML. Whether it's worth going
past that lookup table into ML is an open, testable question: only worth it if
position/intent/content-type interact in ways too tangled for a table.

In [13]:
# If a flat rule ("CTR < X%") were enough, CTR wouldn't vary by position. Show that it does --
# the spread across tiers is the whole reason a single threshold can't work.
print("CTR median by position tier (repeated from Section 2 for this section's argument):")
print(expected_ctr_by_tier)

# Some tiers' medians land on 0.0 (over half their rows are low-impression, zero-click
# pages -- noise, not signal), so a raw first-tier/last-tier ratio can divide by zero.
# Compare the highest and lowest NON-ZERO tier medians instead -- that's the real spread
# a flat rule would have to account for.
nonzero_medians = expected_ctr_by_tier[expected_ctr_by_tier > 0]
if len(nonzero_medians) >= 2:
    ratio = round(nonzero_medians.max() / nonzero_medians.min(), 2)
    print(f"\nRatio of highest to lowest tier median CTR (excluding zero-median tiers): {ratio}")
else:
    print("\nNot enough non-zero tier medians to compute a ratio.")

zero_tiers = expected_ctr_by_tier[expected_ctr_by_tier == 0].index.tolist()
if zero_tiers:
    print(f"Tiers with a zero median (dominated by low-volume, zero-click rows): {zero_tiers}")

CTR median by position tier (repeated from Section 2 for this section's argument):
position_tier
1-3      0.00
4-10     0.16
11-20    0.10
21+      0.00
Name: ctr, dtype: float64

Ratio of highest to lowest tier median CTR (excluding zero-median tiers): 1.6
Tiers with a zero median (dominated by low-volume, zero-click rows): ['1-3', '21+']


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.